In [0]:
from pyspark.sql.functions import *

df_world_bank_populacao_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_populacao")
df_world_bank_fdi_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_fdi")
df_world_bank_eletricidade_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_eletricidade")
df_world_bank_pib_per_capita_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_pib_per_capita")


def eda_test(df_inicial):
    df_duplicata = df_inicial.groupby("iso3", "date").count().filter(col('count') > 1).show(truncate = False)
    #verifica duplicata, como aqui se repete igual os outros dois df, fica bem explicativo :)

    df_nulo = df_inicial.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_inicial.columns
    ]).show(truncate = False)

    df_inicial.filter(col("iso3").isNull()).select("country_name").distinct().show(truncate=False)

    df_real = df_inicial.withColumn("date_formatada", to_date(col("date"), "yyyy")).filter(col("iso3").isNotNull())
    df_limites_por_pais = df_real.groupBy("iso3").agg(
        min("date_formatada").alias("data_inicial"),
        max("date_formatada").alias("data_final")
    )
    df_cenario_ideal = df_limites_por_pais.select(
        "iso3",
        explode(sequence(col("data_inicial"), col("data_final"), expr("INTERVAL 1 YEAR"))).alias("data_esperada")
    ) 
    df_datas_faltantes = df_cenario_ideal.join(
        df_real,
        (df_cenario_ideal["iso3"] == df_real["iso3"]) & 
        (df_cenario_ideal["data_esperada"] == df_real["date_formatada"]),
        "left_anti"
    ).show(truncate = False)

eda_test(df_world_bank_populacao_bronze)
eda_test(df_world_bank_fdi_bronze)
eda_test(df_world_bank_eletricidade_bronze)
eda_test(df_world_bank_pib_per_capita_bronze)